Бибиблиотеки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

подгрузка датасета

In [ ]:
ds = pd.read_csv('Marketing_Data.csv')
print(ds.head())

,youtube,facebook,newspaper,sales
0,84.72,19.20,48.96,12.60
1,351.48,33.96,51.84,25.68
2,135.48,20.88,46.32,14.28
3,116.64,1.80,36.00,11.52
4,318.72,24.00,0.36,20.88
...,...,...,...,...
166,45.84,4.44,16.56,9.12
167,113.04,5.88,9.72,11.64
168,212.40,11.16,7.68,15.36
169,340.32,50.40,79.44,30.60


In [ ]:
ds.hist(figsize=(10, 8), edgecolor='black')
plt.suptitle("Feature Distributions")
plt.savefig('feature_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

распределения признаков

![](feature_histograms.png)

выбросы

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=ds)
plt.title("Boxplot (Outliers Detection)")
plt.savefig('boxplot_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

![](boxplot_outliers.png)

избавление от выбросов

In [ ]:
cols = ['youtube', 'facebook', 'newspaper', 'sales']
mask = (ds[cols] - ds[cols].mean()).abs() <= (3 * ds[cols].std())
ds_clean = ds[mask.all(axis=1)].reset_index(drop=True)

print("Размер исходных данных:", ds.shape)
print("После очистки:", ds_clean.shape)

Исходный размер: 171
Размер после удаления выбросов: 170


разделение на признаки

In [ ]:
X = ds[['youtube', 'facebook', 'newspaper']]
y = ds['sales']

X_clean = ds_clean[['youtube', 'facebook', 'newspaper']]
y_clean = ds_clean['sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42
)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clean, y_clean, test_size=0.4, random_state=42
)

scaler = StandardScaler()
scaler.fit(Xc_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Xc_train_scaled = scaler.transform(Xc_train)
Xc_test_scaled  = scaler.transform(Xc_test)

model = LinearRegression()
model.fit(Xc_train_scaled, yc_train)

y_pred_dirty = model.predict(X_test_scaled)   # тест с выбросами
y_pred_clean = model.predict(Xc_test_scaled)  # тест без выбросов

,youtube,facebook,newspaper,sales
youtube,1.000000,0.086538,0.110759,0.782030
facebook,0.086538,1.000000,0.293425,0.602918
newspaper,0.110759,0.293425,1.000000,0.254987
sales,0.782030,0.602918,0.254987,1.000000


метрики

In [ ]:
def get_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

rmse_dirty, mae_dirty, r2_dirty = get_metrics(y_test, y_pred_dirty)
rmse_clean, mae_clean, r2_clean = get_metrics(yc_test, y_pred_clean)

results = pd.DataFrame({
    "Test data": ["With outliers", "Without outliers"],
    "RMSE": [rmse_dirty, rmse_clean],
    "MAE": [mae_dirty, mae_clean],
    "R2": [r2_dirty, r2_clean]
})

print("\nModel performance on different test sets (trained on clean data):")
print(results)

С выбросами:
  RMSE: 2.1916
  R2: 0.8756
  MAE: 1.5533

Без выбросов:
  RMSE: 1.9361
  R2: 0.8922
  MAE: 1.5027


график обучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# с выбросами
axes[0].scatter(y_test, y_pred_dirty, alpha=0.7, color='tomato', edgecolors='white', s=60)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             linestyle='--', color='black')
axes[0].set_xlabel("Actual Sales")
axes[0].set_ylabel("Predicted Sales")
axes[0].set_title("Test with outliers")
axes[0].text(0.05, 0.93, f'RMSE = {rmse_dirty:.2f}', transform=axes[0].transAxes, fontsize=11)
axes[0].text(0.05, 0.86, f'R² = {r2_dirty:.3f}', transform=axes[0].transAxes, fontsize=11)

# без выбросов
axes[1].scatter(yc_test, y_pred_clean, alpha=0.7, color='steelblue', edgecolors='white', s=60)
axes[1].plot([yc_test.min(), yc_test.max()],
             [yc_test.min(), yc_test.max()],
             linestyle='--', color='black')
axes[1].set_xlabel("Actual Sales")
axes[1].set_ylabel("Predicted Sales")
axes[1].set_title("Test without outliers")
axes[1].text(0.05, 0.93, f'RMSE = {rmse_clean:.2f}', transform=axes[1].transAxes, fontsize=11)
axes[1].text(0.05, 0.86, f'R² = {r2_clean:.3f}', transform=axes[1].transAxes, fontsize=11)

plt.suptitle("One model (trained on clean data) tested on different sets", fontsize=14)
plt.tight_layout()
plt.savefig('actual_vs_predicted_one_model.png', dpi=150, bbox_inches='tight')
plt.show()


![](actual_vs_predicted_one_model.png)